# dθ/dQ versus interpolation mesh

Replaces `plot_dtheta_vs_nk.ipynb` (MnBi2Te4) and `axion_response_vs_nk_plot.ipynb` (Y2Ir2O7). `SWEEPS` maps a
label to a sweep folder; it reads every `nk_*.npz` there, old and new layouts alike. Filled
markers: base-end estimate; open: mode-end estimate (their spread is a linearity test, not a validity test).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))

In [ ]:
MODE_RUN = DATA / "Y2Ir2O7/phonon/GM2-/mode1/Q_0.01A/output/178365bc889/trial_02_Y_p_Ir_d_O_sp/no_displacement/181565bc889"
SWEEPS = {
    ".mmn, own-centre images (old)": MODE_RUN / "nk_sweep_ext_ws_mmn",
    ".mmn, shared base centres (current best)": MODE_RUN / "nk_sweep_mmn_shared_ws",
    "Jae-Mo's files": MODE_RUN / "nk_sweep_ext_ndegen",
    "Jae-Mo's formula, shared centres": MODE_RUN / "nk_sweep_tif_shared_ws",
}
NK_MIN = 6

In [ ]:
def load_sweep(folder):
    rows = []
    for path in folder.glob("nk_*.npz"):
        data = np.load(path, allow_pickle=True)
        rows.append((int(data["nk"]) if "nk" in data.files else int(path.stem[3:]), np.asarray(data["dtheta"]).real))
    rows.sort()
    return np.array([n for n, _ in rows]), np.array([d for _, d in rows])


fig, ax = plt.subplots(figsize=(8, 4))
for i, (label, folder) in enumerate(SWEEPS.items()):
    nks, dtheta = load_sweep(folder)
    keep = nks >= NK_MIN
    ax.plot(nks[keep], dtheta[keep, 0], "o-", color=f"C{i}", label=label)
    ax.plot(nks[keep], dtheta[keep, 1], "o--", color=f"C{i}", mfc="none")
    print(f"{label}: " + ", ".join(f"nk={n}: {d[0]:+.5f}/{d[1]:+.5f}" for n, d in zip(nks, dtheta)))
ax.set(xlabel="interpolation $n_k$", ylabel=r"$d\theta/dQ$ (rad/Å)")
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
plt.show()